# Phase 3 — Modèles ML : Logistic Regression & Naive Bayes
**Auteur : Aymen Ichqarrane | PFA 4ème Année**

- Entrée  : `data_splits.pkl` (produit par Ihssane en Phase 2)
- Sortie  : `models/logistic_regression.pkl`, `models/naive_bayes.pkl`
- Objectif: Entraîner, évaluer et comparer LR vs NB sur 3 classes de sentiment

**Imports**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)
print("Imports OK")

**Chargement des données (depuis le .pkl d'Ihssane)**

In [ ]:
import os

data_path = os.path.join(REPO_PATH, 'data')

X_train = joblib.load(os.path.join(data_path, 'X_train.pkl'))
X_test  = joblib.load(os.path.join(data_path, 'X_test.pkl'))
y_train = pd.read_csv(os.path.join(data_path, 'y_train.csv')).squeeze() # .squeeze() to convert DataFrame to Series if it's a single column
y_test  = pd.read_csv(os.path.join(data_path, 'y_test.csv')).squeeze() # .squeeze() to convert DataFrame to Series if it's a single column

print(f"Train : {X_train.shape[0]:,} avis \u00d7 {X_train.shape[1]:,} features")
print(f"Test  : {X_test.shape[0]:,} avis")
print(f"\nClasses : {sorted(y_train.unique())}")
print(f"\nDistribution train :")
print(pd.Series(y_train).value_counts())

**Fonction d'évaluation**

In [ ]:
def evaluer_modele(nom, modele, X_train, X_test, y_train, y_test):
    """
    Entraîne le modèle, affiche toutes les métriques, retourne un dict de résultats.
    """
    # Entraînement
    modele.fit(X_train, y_train)

    # Prédictions
    y_pred = modele.predict(X_test)

    # Métriques principales
    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred, average='weighted')
    f1_m = f1_score(y_test, y_pred, average='macro')

    print(f"\n{'='*55}")
    print(f"  MODÈLE : {nom}")
    print(f"{'='*55}")
    print(f"  Accuracy         : {acc:.4f}  ({acc*100:.2f}%)")
    print(f"  F1 weighted      : {f1:.4f}")
    print(f"  F1 macro         : {f1_m:.4f}")
    print(f"\n{classification_report(y_test, y_pred)}")

    return {
        'nom': nom,
        'modele': modele,
        'y_pred': y_pred,
        'accuracy': acc,
        'f1_weighted': f1,
        'f1_macro': f1_m
    }

print("Fonction evaluer_modele() prête")

**Logistic Regression (avec GridSearch)**

In [ ]:
lr_base = LogisticRegression(
    multi_class='multinomial',   # 3 classes → softmax
    solver='lbfgs',              # optimiseur adapté au multinomial
    max_iter=1000,               # assez d'itérations pour converger
    random_state=42
)

# GridSearch : on cherche le meilleur C parmi ces valeurs
param_grid_lr = {'C': [0.01, 0.1, 1, 10, 100]}

gs_lr = GridSearchCV(
    lr_base,
    param_grid_lr,
    cv=5,                  # 5-fold cross-validation
    scoring='f1_weighted', # on optimise le F1 (pas juste l'accuracy)
    n_jobs=-1,             # utiliser tous les cœurs disponibles
    verbose=0
)

print("GridSearch LR en cours...")
gs_lr.fit(X_train, y_train)

print(f"Meilleur C trouvé : {gs_lr.best_params_['C']}")
print(f"F1 CV (train)     : {gs_lr.best_score_:.4f}")

# Évaluer le meilleur modèle sur le test set
resultats_lr = evaluer_modele(
    "Logistic Regression (C=" + str(gs_lr.best_params_['C']) + ")",
    gs_lr.best_estimator_,
    X_train, X_test, y_train, y_test
)

**Naive Bayes (avec GridSearch)**

In [ ]:
nb_base = MultinomialNB()

param_grid_nb = {'alpha': [0.01, 0.1, 0.5, 1.0, 2.0]}

gs_nb = GridSearchCV(
    nb_base,
    param_grid_nb,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1
)

print("GridSearch NB en cours...")
gs_nb.fit(X_train, y_train)

print(f"Meilleur alpha : {gs_nb.best_params_['alpha']}")
print(f"F1 CV (train)  : {gs_nb.best_score_:.4f}")

resultats_nb = evaluer_modele(
    "Naive Bayes (alpha=" + str(gs_nb.best_params_['alpha']) + ")",
    gs_nb.best_estimator_,
    X_train, X_test, y_train, y_test
)

**Matrices de confusion (visualisation)**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
classes = ['NEGATIF', 'NEUTRE', 'POSITIF']

for ax, resultats in zip(axes, [resultats_lr, resultats_nb]):
    cm = confusion_matrix(y_test, resultats['y_pred'], labels=classes)

    # Normaliser pour avoir des % (plus lisible quand les classes sont déséquilibrées)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    sns.heatmap(
        cm_pct, annot=True, fmt='.1f', cmap='Blues',
        xticklabels=classes, yticklabels=classes,
        ax=ax, cbar=False, linewidths=0.5
    )
    ax.set_title(f"{resultats['nom']}\nAcc={resultats['accuracy']:.3f} | F1={resultats['f1_weighted']:.3f}",
                 fontsize=11)
    ax.set_xlabel('Prédit')
    ax.set_ylabel('Réel')

plt.suptitle('Matrices de confusion — Phase 3 Aymen', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
comparaison = pd.DataFrame([
    {
        'Modèle'       : resultats_lr['nom'],
        'Accuracy'     : f"{resultats_lr['accuracy']:.4f}",
        'F1 weighted'  : f"{resultats_lr['f1_weighted']:.4f}",
        'F1 macro'     : f"{resultats_lr['f1_macro']:.4f}",
    },
    {
        'Modèle'       : resultats_nb['nom'],
        'Accuracy'     : f"{resultats_nb['accuracy']:.4f}",
        'F1 weighted'  : f"{resultats_nb['f1_weighted']:.4f}",
        'F1 macro'     : f"{resultats_nb['f1_macro']:.4f}",
    },
])
print(comparaison.to_string(index=False))

In [ ]:
models_path = os.path.join(REPO_PATH, 'models')
os.makedirs(models_path, exist_ok=True)

# Sauvegarder les deux modèles
joblib.dump(gs_lr.best_estimator_, os.path.join(models_path, 'logistic_regression.pkl'))
print("logistic_regression.pkl sauvegardé")

joblib.dump(gs_nb.best_estimator_, os.path.join(models_path, 'naive_bayes.pkl'))
print("naive_bayes.pkl sauvegardé")

# Sauvegarder aussi les résultats pour la comparaison finale
joblib.dump({
    'logistic_regression': resultats_lr,
    'naive_bayes'        : resultats_nb
}, os.path.join(models_path, 'resultats_aymen.pkl'))

print("\nPhase 3 Aymen — TERMINÉE ✓")
print(f"Meilleur modèle Aymen : ", end="")
best = max([resultats_lr, resultats_nb], key=lambda r: r['f1_weighted'])
print(f"{best['nom']} (F1={best['f1_weighted']:.4f})")